# AMD AAI Workshop 01: Multi-Turn RL Training – Qwen3-1.7B + GRPO (Miles + Megatron)

**Hardware:** MI300X (ROCm)  
**Workshop focus:** Multi-turn RL with GRPO on CodeContests-style tasks using Miles + Megatron.

---

## UnicornForge AI Integration (Hackathon Value)

**Originality:** Instead of classic fine-tuning, we use **multi-turn RL** to train a model that iteratively improves startup briefs.

**Business Value:** The model becomes an "AI Co-Founder" – it takes a raw brief from UnicornForge and refines it over several turns for clarity, market fit, and demo potential (using our SuccessPredictor as a reward signal).

**Presentation + Video:** We demonstrate training on real MI300X in the AMD cloud + results: briefs with higher scores than pure dataset/Fireworks fallback. The full stack is Fireworks AI (inference & data) + AMD (training & acceleration).

**5-Star Hackathon Plan:**
- Originality: RL instead of plain SFT
- Business: RL directly optimizes for our success model + hackathon judges
- Video: "Trained on AMD MI300X in 30 min → better briefs in 5s"

---

## 1. Environment (ROCm on MI300X)

```python
import torch
import os

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    try:
        device_name = torch.cuda.get_device_name(0)
    except:
        device_name = "AMD GPU (ROCm)"
    print("Device:", device_name)
    print("ROCm/HIP visible devices:", os.environ.get("ROCR_VISIBLE_DEVICES", "all"))
    print("GPU count:", torch.cuda.device_count())
```

**Oczekiwany output na Twojej instancji:**
```
PyTorch: 2.9.1+gitff65f5b
CUDA available: True
Device: 
ROCm/HIP visible devices: all
```

---

## 2. Installation / Access to Workshop Tools

The AMD AAI instance should have the following available:

```bash
# Check if you have Miles + Megatron
python -c "import miles; print(miles.__file__)"
python -c "import megatron; print('Megatron OK')"
```

If not present, the workshop likely provides installation scripts.

---

## 3. Task Adaptation: CodeContests → Startup Brief Refinement

Instead of generating code for a programming problem, the model will refine a startup brief over several turns.

**Reward model (very important for the hackathon):**
- Our trained `SuccessPredictor` (PyTorch MLP)
- + simple heuristics (length, concreteness, presence of sections)

```python
# Load our predictor z projektu
import sys
sys.path.append("/home/.../UvicornForge-AI/backend")  # adjust path

from ml.predictor import SuccessPredictor
from ml.brief_generator import generate_dataset_brief

predictor = SuccessPredictor()
print("SuccessPredictor ready:", predictor.ready)

def reward_fn(brief_text: str, original_idea: str) -> float:
    """Simple reward – the higher the score from our model, the better."""
    # W prawdziwym treningu: parsujemy brief i liczymy score
    # Na razie simulation
    score = 5.0
    try:
        # You can tu plug in our model
        # result = predictor.predict_from_payload(project_idea=original_idea)
        # score = result.score if result else 5.0
        pass
    except:
        pass
    # Bonus za length i structure
    if len(brief_text) > 400: score += 0.5
    if "Problem:" in brief_text and "Solution:" in brief_text: score += 0.8
    return score
```

---

## 4. Struktura treningu GRPO (zgodnie z warsztatem)

Typowy flow z warsztatu (dostosowany):

```python
# from miles.rl import GRPOConfig, GRPOTrainer
# from megatron import initialize_megatron

# config = GRPOConfig(
#     model_name="Qwen/Qwen3-1.7B",
#     max_turns=4,                    # multi-turn refinement
#     rollout_batch_size=8,
#     ...
# )

# trainer = GRPOTrainer(config)

# Dla demonstracji – loop pseudo-treningowa
for step in range(3):
    # 1. Sample trajectories (model generuje kolejne wersje briefu)
    # 2. Compute rewards using ourego SuccessPredictor
    # 3. GRPO update (Miles + Megatron)
    print(f"Step {step}: GRPO update on MI300X (simulation)")
    
print("RL training loop gotowy do uruchomienia na prawdziwych danych.")
```

---

## 5. Preparing Training Data from UnicornForge (Fireworks + AMD Stack)

Use our dataset + Fireworks API (key from .env) to generate high-quality synthetic briefs as "problems" for RL.

Then train the refiner on AMD MI300X.

```python
import os
import pandas as pd
from openai import OpenAI  # Fireworks is OpenAI-compatible

# Load Fireworks key from .env
fireworks_key = os.getenv("FIREWORKS_API_KEY") or os.getenv("fireworks_key")
fw_client = OpenAI(
    api_key=fireworks_key,
    base_url="https://api.fireworks.ai/inference/v1"
)

def generate_initial_brief_with_fireworks(idea: str) -> str:
    """Use Fireworks to generate initial brief (part of the full stack)."""
    response = fw_client.chat.completions.create(
        model="accounts/fireworks/models/qwen2p5-72b-instruct",  # or any Fireworks model
        messages=[{"role": "user", "content": f"Generate a structured startup brief for: {idea}"}],
        max_tokens=800
    )
    return response.choices[0].message.content

# Load dataset
df = pd.read_csv("/path/to/global_startup_success_dataset.csv").head(50)

training_tasks = []
for _, row in df.iterrows():
    idea = str(row.get("idea", row.get("Startup Name", "AI tool for ...")))
    brief_v0 = generate_initial_brief_with_fireworks(idea)  # Fireworks for data
    training_tasks.append({
        "idea": idea,
        "initial_brief": brief_v0,
        "target_score": float(row.get("Success Score", 5))
    })

print(f"Prepared {len(training_tasks)} RL tasks using Fireworks for data generation.")
```

---

## 6. Uruchomienie na chmurze AMD

On the `hf-367-9ecc9fb2` instance:

1. Skopiuj ten notebook
2. Skopiuj folder `backend/ml` + `trained_models`
3. Uruchom cells z prawdziwym `GRPOTrainer` z warsztatu
4. Po treningu zapisz checkpoint Qwen → use w backendzie jako `AMD_RL_Refiner`

---

## 7. Backend Integration (after the workshop)

```python
# W services/brief_service.py lub nowym amd_rl_refiner.py
class AMDRLBriefRefiner:
    def __init__(self):
        # load wytrenowany Qwen z MI300X
        ...

    def refine(self, initial_brief: dict) -> dict:
        # multi-turn refinement
        ...
        return improved_brief
```

In the `generate-brief` endpoint:
- First dataset/Fireworks
- Then RL refiner on MI300X → higher score

---

## Dla Video (5 min)

**Video Sequence (5 min):**
1. Show raw brief from dataset (score 5.2)
2. Run RL refiner (training on MI300X in background or pre-recorded)
3. After 3 turns → score 7.8 + better language
4. "Trained on AMD MI300X using Miles + Megatron GRPO – full Fireworks + AMD stack"

To daje bardzo strong originality.

---

**Next kroki dla Ciebie:**

1. Wklej ten notebook na your instance.
2. Dostosuj ścieżki do projektu.
3. Gdy będziesz miał dostęp do kodu Miles/Megatron z warsztatu – zamień pseudokod na prawdziwy.
4. Powiedz, który warsztat robimy jako następny (02, 03, 07 itd.), to przygotuję szkielet z such samą jakością.

Chcesz version tego notatnika z większą ilością konkretnego kodu pod Qwen + GRPO (nawet jeśli częściowo placeholder)? Mogę rozwinąć.
